# Supplementary Data Table 3

Environment: env1

In [ ]:
import pandas as pd
import pickle
import os
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score, roc_curve, auc, precision_recall_curve
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.metrics import confusion_matrix
from scipy.stats import sem
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve
)



In [3]:
def classification_metrics(cases_pos, cases_neg, controls_pos, controls_neg):
    """
    Compute classification metrics from case/control predicted counts.
 
    Parameters
    ----------
    cases_pos    : int | float  — cases predicted positive    (True Positives)
    cases_neg    : int | float  — cases predicted negative    (False Negatives)
    controls_pos : int | float  — controls predicted positive (False Positives)
    controls_neg : int | float  — controls predicted negative (True Negatives)
 
    Returns
    -------
    dict with keys:
        TP, FP, TN, FN,
        TPR  (sensitivity / recall),
        FPR  (fall-out),
        PPV  (precision),
        NPV,
        F1
    """
    TP = cases_pos
    FN = cases_neg
    FP = controls_pos
    TN = controls_neg
 
    TPR = TP / (TP + FN) if (TP + FN) > 0 else float("nan")   # sensitivity
    FPR = FP / (FP + TN) if (FP + TN) > 0 else float("nan")   # fall-out
    PPV = TP / (TP + FP) if (TP + FP) > 0 else float("nan")   # precision
    NPV = TN / (TN + FN) if (TN + FN) > 0 else float("nan")
    F1  = (2 * PPV * TPR) / (PPV + TPR) if (PPV + TPR) > 0 else float("nan")
 
    return {
        "TP":  TP,
        "FP":  FP,
        "TN":  TN,
        "FN":  FN,
        "TPR": TPR,   # sensitivity / recall
        "FPR": FPR,
        "PPV": PPV,   # precision
        "NPV": NPV,
        "F1":  F1,
    }

In [4]:
all_mets = pd.read_csv("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/rev_strat_pec_hc_pw/run_20260407_stable/meta_probs_auc_0.5/all_pw_metrics.csv")
clinical = pd.read_csv("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/clinical_ml_feats_20260430.csv")
result = all_mets.loc[all_mets.groupby('pw')['auc'].idxmax(), ['pw', 'hp', 'auc']]
result.head()

,pw,hp,auc
0,1,"{n_estimators_250,max_depth_3,learning_rate_0....",0.760809
324,2,"{n_estimators_250,max_depth_2,learning_rate_0....",0.663553
650,3,"{n_estimators_100,max_depth_3,learning_rate_0....",0.899518
982,4,"{n_estimators_250,max_depth_3,learning_rate_0....",0.793789
1296,5,"{n_estimators_250,max_depth_1,learning_rate_0....",0.715025


In [5]:
cu_all_probs = []
for pw in range(1, 6):
    hp = result[result["pw"] == pw]["hp"].iloc[0]
    path = f"/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/rev_strat_pec_hc_pw{pw}/run_20260407_stable/meta_probs_auc_0.5/XGBoost/{hp}_oof.pkl"
    with open(path, "rb") as f:
        data = pickle.load(f)
        cu_all_probs.append(data[1])
cu_all_probs = pd.DataFrame(cu_all_probs)

In [6]:
cu_all_probs = cu_all_probs.T
cu_all_probs["id"] = cu_all_probs.index
cu_all_probs = cu_all_probs.reset_index(drop=True)
cu_all_probs = cu_all_probs.merge(clinical, how="left", on='id')
# cu_all_probs = cu_all_probs.merge(fmf_loo, how="left", on='id')
# cu_all_probs = cu_all_probs.merge(clinical_unproc, how="left", on='id')

In [7]:
with open("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/final_pec_20251017.pkl", "rb") as f:
    pec = pickle.load(f)
with open("/gpfs/data/shenhavlab/users/ca3261/Visionary_AI/data/rev_test_pec_cases_20260406.pkl", "rb") as f:
    pec_test = pickle.load(f)
all_pec = pec + pec_test
cu_all_probs["label"] = [int(id in all_pec) for id in cu_all_probs["id"]]

cu_nsf = pd.read_pickle('/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/final_nsf_20251028.pkl')
cu_sf = pd.read_pickle('/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/final_sf_20251028.pkl')
cu_eope = pd.read_pickle('/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/final_eope_20251017.pkl')
cu_lope = pd.read_pickle('/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/final_lope_20251028.pkl')

cases = cu_nsf+cu_sf+cu_eope+cu_lope
cases = list(set(cases))

rename_dict = {}

for i in range(6):
    rename_dict[i] = f"pw_{i + 1}"

cu_all_probs = cu_all_probs.rename(columns=rename_dict)

In [8]:
cu_all_probs['avg_prob'] = cu_all_probs[['pw_1', 'pw_2', 'pw_3', 'pw_4', 'pw_5']].mean(axis=1)

In [10]:
reweight = cu_all_probs.copy()
cols = ["Combo"]
reweight["Combo"] = (reweight["Past PEC"] * 0.35 + 0.15 *((reweight["First Preg"]) * (reweight["Age at enrollment"] > 37).astype(int)))

In [16]:
cases_types = {'sf':cu_sf,'nsf':cu_nsf,'eope':cu_eope,'lope':cu_lope,'pec':cases}

for k,ct in cases_types.items():

    print(k)

    t = reweight[(reweight['id'].isin(ct))|(~reweight['id'].isin(cases))]
    cols = [f'pw_{i}' for i in range(1,6)]
    cols.append('Combo')
    t['pred'] = t[cols].mean(axis=1)

    reweight['pred'] = reweight[cols].mean(axis=1)

    eval_col = 'pred'

    fpr, tpr, thresholds = roc_curve(t['label'], t[eval_col])
    roc_df = pd.DataFrame({'fpr': fpr, 'tpr': tpr, 'threshold': thresholds})
    whole_fpr, whole_tpr, whole_thresholds = roc_curve(reweight['label'], reweight[eval_col])
    whole_roc_df = pd.DataFrame({'fpr': whole_fpr, 'tpr': whole_tpr, 'threshold': whole_thresholds})

    target=0.1

    row = whole_roc_df.iloc[(whole_roc_df['fpr'] - target).abs().argsort()[0]]
    thresh = row['threshold']
    
    preds = (t[eval_col] >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(t['label'], preds).ravel()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    print(f"\nFPR ~{target:.0%} (threshold={thresh:.3f}):")
    print(f"  TPR/Recall: {recall:.3f}  |  Precision: {precision:.3f}")
    print(f"  TP={tp}, FP={fp}, TN={tn}, FN={fn}")
    
    cm = classification_metrics(tp, fn, fp, tn)
    for i in cm:
        print(f'{i}: {cm[i]}')

    t['label'].sum()

    classification_metrics(tp, fn, fp, tn)
    print()

sf

FPR ~10% (threshold=0.406):
  TPR/Recall: 0.658  |  Precision: 0.185
  TP=25, FP=110, TN=1024, FN=13
TP: 25
FP: 110
TN: 1024
FN: 13
TPR: 0.6578947368421053
FPR: 0.09700176366843033
PPV: 0.18518518518518517
NPV: 0.9874638379942141
F1: 0.28901734104046245

nsf

FPR ~10% (threshold=0.406):
  TPR/Recall: 0.500  |  Precision: 0.068
  TP=8, FP=110, TN=1024, FN=8
TP: 8
FP: 110
TN: 1024
FN: 8
TPR: 0.5
FPR: 0.09700176366843033
PPV: 0.06779661016949153
NPV: 0.9922480620155039
F1: 0.11940298507462686

eope

FPR ~10% (threshold=0.406):
  TPR/Recall: 0.600  |  Precision: 0.098
  TP=12, FP=110, TN=1024, FN=8
TP: 12
FP: 110
TN: 1024
FN: 8
TPR: 0.6
FPR: 0.09700176366843033
PPV: 0.09836065573770492
NPV: 0.9922480620155039
F1: 0.16901408450704225

lope

FPR ~10% (threshold=0.406):
  TPR/Recall: 0.618  |  Precision: 0.160
  TP=21, FP=110, TN=1024, FN=13
TP: 21
FP: 110
TN: 1024
FN: 13
TPR: 0.6176470588235294
FPR: 0.09700176366843033
PPV: 0.16030534351145037
NPV: 0.9874638379942141
F1: 0.25454545454545

/gpfs/data/shenhavlab/users/Daniel/.conda/envs/env1/lib/python3.7/site-packages/ipykernel_launcher.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  # Remove the CWD from sys.path while we load stuff.
/gpfs/data/shenhavlab/users/Daniel/.conda/envs/env1/lib/python3.7/site-packages/ipykernel_launcher.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  # Remove the CWD from sys.path while we load stuff.
/gpfs/data/shenhavlab/users/Daniel/.conda/envs/env1/lib/python3.7/site-packages/ipykernel_launcher.py:10